In [ ]:
'''
James Jolly
Created: May 12, 2025
Purpose: Used to test model architecture layout without running full training
'''


In [1]:

# TensorFlow, keras, np
import tensorflow as tf
from tensorflow import keras
import numpy as np
import time
import sys

from io import UnsupportedOperation
import pickle as pkl


# Add shared location for auxillary functions
sys.path.insert(1, './../AuxillaryFunctions')

# import personal Functions
from GenerateClassDataFuncs import GenerateDataAndClasses_ClemCafe_WinCnt
from GenerateClassDataFuncs import GenerateDataAndClasses_OREBA_WinCnt

from ModelSelectFuncs_Search import ModelSelect



print("Finished Importing Libraries.")

Finished Importing Libraries.


In [6]:
MODEL_ARCH_FLAG = 1333

sample_length = 225 # 15 sec @ 15 Hz
total_axes = 6 # start with one handed
# total_axes = 12 # Test for Two handed


model, NUM_EPOCHS = ModelSelect(MODEL_ARCH_FLAG, sample_length, total_axes)

Arch 1333
Architecture Scan of A300 architecture.
Added Dropout layers to model.
Using 24 Filters as base in ConvKernels
Using 24 Nodes in LSTM layers
Using 24 Neurons in Dense Hidden Layer


In [7]:



FilterIdx = int((MODEL_ARCH_FLAG / 100) % 10) # Hundreds Digit
LstmIdx = int((MODEL_ARCH_FLAG / 10) % 10) # Tens Digit
DenseIdx = int(MODEL_ARCH_FLAG % 10) # Ones Digit


BaseFilterOptions = [8, 12, 16, 24, 32, 40, 48, 64]
BaseFilterCnt = BaseFilterOptions[FilterIdx]

LstmNodesOptions = [8, 12, 16, 24, 32, 40, 48, 64]
LstmNodeCnt = BaseFilterOptions[LstmIdx]

DenseNeuronOptions = [8, 12, 16, 24, 32, 40, 48, 64]
DenseNeuronCnt = DenseNeuronOptions[DenseIdx]

print("Using {:d} Filters as base in ConvKernels".format(BaseFilterCnt))
print("Using {:d} Nodes in LSTM layers".format(LstmNodeCnt))
print("Using {:d} Neurons in Dense Hidden Layer".format(DenseNeuronCnt))


Using 24 Filters as base in ConvKernels
Using 24 Nodes in LSTM layers
Using 24 Neurons in Dense Hidden Layer


In [8]:

model = keras.Sequential([
	keras.layers.Conv1D(input_shape=(sample_length,total_axes,),
		filters=BaseFilterCnt,
		kernel_size=3, #initial layer 1/4 of a second
		strides=1,
		activation='relu'),
	keras.layers.BatchNormalization(),
	keras.layers.Dropout(0.2),

	keras.layers.Conv1D(filters=BaseFilterCnt,
		kernel_size=3,
		activation='relu'),
	keras.layers.BatchNormalization(),
	keras.layers.Dropout(0.2),


	keras.layers.MaxPool1D(),


	keras.layers.Conv1D(filters=2*BaseFilterCnt,
		kernel_size=3,
		activation='relu'),
	keras.layers.BatchNormalization(),
	keras.layers.Dropout(0.2),

	keras.layers.Conv1D(filters=2*BaseFilterCnt,
		kernel_size=3,
		activation='relu'),
	keras.layers.BatchNormalization(),
	keras.layers.Dropout(0.2),


	keras.layers.MaxPool1D(),


	keras.layers.Conv1D(filters=3*BaseFilterCnt,
		kernel_size=5,
		activation='relu'),
	keras.layers.BatchNormalization(),
	keras.layers.Dropout(0.2),

	keras.layers.Conv1D(filters=3*BaseFilterCnt,
		kernel_size=5,
		activation='relu'),
	keras.layers.BatchNormalization(),
	keras.layers.Dropout(0.2),



#    keras.layers.SimpleRNN(units=128,activation='relu'),
#    keras.layers.SimpleRNN(units=128,activation="relu",return_sequences=True),

	###  SECOND PHASE: TIME MEMORY OF GESTURES  ### 
	#####  TAKEN FROM OREBA REIMPLEMENTATION  #####
	keras.layers.LSTM(LstmNodeCnt, return_sequences=True,
		activation="tanh",recurrent_activation="hard_sigmoid"),
	keras.layers.LSTM(LstmNodeCnt, 
		activation="tanh",recurrent_activation="hard_sigmoid"),
	###  END OF BORROWED REIMPLEMENTATION  ###

	keras.layers.Flatten(),  # must flatten to feed dense layer
	keras.layers.Dropout(0.2),
	keras.layers.Dense(DenseNeuronCnt),
	keras.layers.Dropout(0.2),
	keras.layers.Dense(1)
	])

print("Finished Defining Model Arch")

Finished Defining Model Arch


In [9]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv1d_18 (Conv1D)           (None, 223, 24)           456       
_________________________________________________________________
batch_normalization_18 (Batc (None, 223, 24)           96        
_________________________________________________________________
dropout_24 (Dropout)         (None, 223, 24)           0         
_________________________________________________________________
conv1d_19 (Conv1D)           (None, 221, 24)           1752      
_________________________________________________________________
batch_normalization_19 (Batc (None, 221, 24)           96        
_________________________________________________________________
dropout_25 (Dropout)         (None, 221, 24)           0         
_________________________________________________________________
max_pooling1d_6 (MaxPooling1 (None, 110, 24)          

In [19]:
Model: "sequential_4"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
conv1d_24 (Conv1D)           (None, 223, 32)           608       
_________________________________________________________________
batch_normalization_24 (Batc (None, 223, 32)           128       
_________________________________________________________________
dropout_32 (Dropout)         (None, 223, 32)           0         
_________________________________________________________________
conv1d_25 (Conv1D)           (None, 221, 32)           3104      
_________________________________________________________________
batch_normalization_25 (Batc (None, 221, 32)           128       
_________________________________________________________________
dropout_33 (Dropout)         (None, 221, 32)           0         
_________________________________________________________________
max_pooling1d_8 (MaxPooling1 (None, 110, 32)           0         
_________________________________________________________________
conv1d_26 (Conv1D)           (None, 108, 64)           6208      
_________________________________________________________________
batch_normalization_26 (Batc (None, 108, 64)           256       
_________________________________________________________________
dropout_34 (Dropout)         (None, 108, 64)           0         
_________________________________________________________________
conv1d_27 (Conv1D)           (None, 106, 64)           12352     
_________________________________________________________________
batch_normalization_27 (Batc (None, 106, 64)           256       
_________________________________________________________________
dropout_35 (Dropout)         (None, 106, 64)           0         
_________________________________________________________________
max_pooling1d_9 (MaxPooling1 (None, 53, 64)            0         
_________________________________________________________________
conv1d_28 (Conv1D)           (None, 49, 128)           41088     
_________________________________________________________________
batch_normalization_28 (Batc (None, 49, 128)           512       
_________________________________________________________________
dropout_36 (Dropout)         (None, 49, 128)           0         
_________________________________________________________________
conv1d_29 (Conv1D)           (None, 45, 128)           82048     
_________________________________________________________________
batch_normalization_29 (Batc (None, 45, 128)           512       
_________________________________________________________________
dropout_37 (Dropout)         (None, 45, 128)           0         
_________________________________________________________________
lstm_8 (LSTM)                (None, 45, 32)            20608     
_________________________________________________________________
lstm_9 (LSTM)                (None, 32)                8320      
_________________________________________________________________
flatten_4 (Flatten)          (None, 32)                0         
_________________________________________________________________
dropout_38 (Dropout)         (None, 32)                0         
_________________________________________________________________
dense_8 (Dense)              (None, 24)                792       
_________________________________________________________________
dropout_39 (Dropout)         (None, 24)                0         
_________________________________________________________________
dense_9 (Dense)              (None, 1)                 25        
=================================================================
Total params: 176,945
Trainable params: 176,049
Non-trainable params: 896
_________________________________________________________________

SyntaxError: invalid syntax (<ipython-input-19-52df45fca4ef>, line 3)